In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

np.random.seed(42)

# Synthetic dataset
n = 500
df = pd.DataFrame({
    'customer_id': [f'C-{i:04d}' for i in range(n)],
    'signup_date': pd.date_range('2020-01-01', periods=n, freq='D').astype(str),
    'notes': [np.random.choice([np.nan, 'customer wants to cancel', 'account active', 'needs follow-up'], p=[0.87, 0.07, 0.04, 0.02]) for _ in range(n)],
    'churned': np.random.randint(0, 2, n),
})

print(f"Dataset shape: {df.shape}")
print(f"Notes nulls: {df['notes'].isnull().sum()}")

## Bug 1: Using pd.Timestamp.today() as reference date (non-reproducibility)

In [ ]:
# --- BUGGY CODE ---
df['signup_date'] = pd.to_datetime(df['signup_date'])
df['tenure_days'] = (pd.Timestamp.today() - df['signup_date']).dt.days
df['is_veteran'] = (df['tenure_days'] > 365).astype(int)

print(f"Tenure days (today):")
print(df['tenure_days'].describe())
print(f"\nExpected: Tenure varies based on today's date (BUG)")
print(f"Explanation: Write here why this is non-reproducible.")

**Fix:**

In [ ]:
# Use a fixed reference date (last date in the dataset)
reference_date = df['signup_date'].max() + pd.Timedelta(days=1)
df['tenure_days'] = (reference_date - df['signup_date']).dt.days
df['is_veteran'] = (df['tenure_days'] > 365).astype(int)

print(f"Tenure days (fixed reference {reference_date.date()}):")
print(df['tenure_days'].describe())
print(f"\nReproducible: same result every time this code runs.")

## Bug 2: TfidfVectorizer fit on full dataset before split (leakage)

In [ ]:
# --- BUGGY CODE ---
notes_filled = df['notes'].fillna('')

# Fit on ALL data (leakage!)
vec = TfidfVectorizer(max_features=10, stop_words='english')
notes_matrix = vec.fit_transform(notes_filled)

# Then split
X_train, X_test, y_train, y_test = train_test_split(
    notes_matrix, df['churned'], test_size=0.2, random_state=42
)

print(f"Vectorizer vocabulary (IDF from full dataset):")
print(vec.get_feature_names_out())
print(f"\nExplanation: Write here which statistics from the test set are now in the training data.")

**Fix:**

In [ ]:
# Correct: split FIRST
X_train_idx, X_test_idx, y_train, y_test = train_test_split(
    np.arange(len(df)), df['churned'], test_size=0.2, random_state=42
)

notes_train = df.loc[X_train_idx, 'notes'].fillna('')
notes_test = df.loc[X_test_idx, 'notes'].fillna('')

# Fit vectorizer on training notes ONLY
vec = TfidfVectorizer(max_features=10, stop_words='english')
X_train = vec.fit_transform(notes_train)
X_test = vec.transform(notes_test)  # transform only

print(f"Vectorizer fit on training notes only")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"No leakage: test set IDF not in training vocabulary.")

## Bug 3: Building keyword list from full dataset target labels (target leakage)

In [ ]:
# --- BUGGY CODE ---
# AI-generated: find high-signal terms by looking at churned customers across FULL dataset
high_churn_notes = df[df['churned'] == 1]['notes'].dropna()
word_freq = pd.Series(' '.join(high_churn_notes).split()).value_counts()
top_terms = '|'.join(word_freq.head(5).index)

print(f"Top terms from churned customers (full dataset): {top_terms}")

df['at_risk_flag'] = df['notes'].str.contains(top_terms, case=False, na=False).astype(int)
print(f"\nat_risk_flag correlation with churn:")
print(f"Churn rate when at_risk_flag=1: {df[df['at_risk_flag']==1]['churned'].mean():.2%}")
print(f"Churn rate when at_risk_flag=0: {df[df['at_risk_flag']==0]['churned'].mean():.2%}")
print(f"\nExplanation: Write here why this is target leakage.")

**Fix:**

In [ ]:
# Correct: build keyword list from TRAINING data only
X_train_idx, X_test_idx, y_train, y_test = train_test_split(
    np.arange(len(df)), df['churned'], test_size=0.2, random_state=42
)

# Find terms only from training churned notes
high_churn_notes_train = df.loc[X_train_idx]
high_churn_notes_train = high_churn_notes_train[high_churn_notes_train['churned'] == 1]['notes'].dropna()

if len(high_churn_notes_train) > 0:
    word_freq_train = pd.Series(' '.join(high_churn_notes_train).split()).value_counts()
    top_terms_train = '|'.join(word_freq_train.head(5).index)
    print(f"Top terms from training churned notes: {top_terms_train}")
    
    # Apply to BOTH train and test
    df['at_risk_flag'] = df['notes'].str.contains(top_terms_train, case=False, na=False).astype(int)
    print(f"\nNo target leakage: keyword list built from training data only.")
else:
    print("No churned customers in training set to build keyword list from.")

## Summary check

In [ ]:
print("Bug 1: pd.Timestamp.today() → Fixed reference date (reproducibility)")
print("Bug 2: TfidfVectorizer fit on full data → Split first, fit on train only (prevent leakage)")
print("Bug 3: Keyword list from full target → Build from training data only (prevent target leakage)")
print("\nAll bugs fixed. Ready for the final lesson on handling data leakage.")